# mweralign

In [1]:
import os
import json

In [4]:
log_path = '/ckpts/infinisst/yodas_5kh_v2_m12_stage2_7b_1node/checkpoints/infinisst/acl6060/cache200-2000/instances.log'
with open(log_path, 'r') as f:
    logs = [json.loads(line) for line in f.readlines() if line.strip()]

In [9]:
import yaml

yaml_path = '/data/st/acl_6060/acl_6060/dev/ACL.ACLdev2023.en-xx.gold_segments.yaml'
with open(yaml_path, 'r') as f:
    manifest = yaml.load(f, Loader=yaml.FullLoader)

In [11]:
ref_path = '/data/st/acl_6060/acl_6060/dev/text/txt/ACL.6060.dev.en-xx.zh.txt'
with open(ref_path, 'r') as f:
    refs = [line.strip() for line in f.readlines() if line.strip()]

In [15]:
with open('hyp.txt', 'w') as w:
    w.write(logs[0]['prediction'])
with open('ref.txt', 'w') as w:
    w.write('\n'.join([ref for idx, ref in enumerate(refs) if manifest[idx]['wav'] == manifest[0]['wav']]))

# test sampling

In [1]:
from vllm import LLM, SamplingParams, ModelRegistry
from vllm.model_executor.models.sqwen2 import SQwen2ForConditionalGeneration

ModuleNotFoundError: No module named 'vllm'

In [ ]:

ModelRegistry.register_model("SQwen2ForConditionalGeneration", SQwen2ForConditionalGeneration)
llm = LLM(
    model="/ckpts/infinisst/yodas_5kh_v2_m12_stage2_7b_1node/checkpoints/step=2000-last.ckpt.hf/",
    enforce_eager=True,
    enable_prefix_caching=True,
)
CODE2LANG = {
    'zh': 'Chinese',
    'en': 'English',
    'ja': 'Japanese',
    'ko': 'Korean',
    'fr': 'French',
    'de': 'German',
    'es': 'Spanish',
}

INSTRUCTION = "Translate the following speech from {} to {}.".format(CODE2LANG['en'], CODE2LANG['zh'])

import pandas as pd
df = pd.read_parquet("/data/asr/yodas/npy/parakeet-tdt-0.6b-v2_robust60-1120_langid_zh_blaser2.0-qe3.0_metricx-qe4.0_simalign/en000/manifest.parquet")

data = [dict(df.iloc[idx]) for idx in range(16)]

import torch
import numpy as np
audio_embs = [torch.from_numpy(np.load(data[idx]['audio_npy_path'], mmap_mode='r')[data[idx]['audio_npy_row']].copy()) for idx in range(16)]

sampling_params = SamplingParams(top_p=0.9)
messages = [
    [
        {
            "role": "system",
            "content": INSTRUCTION,
        },
        {
            "role": "user",
            "content": "<|video_pad|>" * 14,
        }
    ]
    for i in range(16)
]
tokenizer = llm.get_tokenizer()

translations = [[] for _ in range(16)]

from tqdm.notebook import tqdm

for i in tqdm(range(60)):
    requests = []
    for j in range(16):
        message = messages[j]
        prompt = tokenizer.apply_chat_template(
            message, 
            tokenize=False,
            add_generation_prompt=True,
        )
        request = {
            "prompt": prompt,
            "multi_modal_data": {
                "audio": [audio_embs[j][k * 14 : (k + 1) * 14] for k in range(i + 1)]
            }
        }
        requests.append(request)
    outputs = llm.generate(requests, sampling_params, use_tqdm=False)
    for j in range(16):
        translations[j].append(outputs[j].outputs[0].text)
        messages[j].append({
            "role": "assistant",
            "content": translations[j][i],
        })
        messages[j].append({
            "role": "user",
            "content": "<|video_pad|>" * 14,
        })

# test reward

In [24]:
from comet import download_model, load_from_checkpoint

In [27]:
model_path = download_model("Unbabel/XCOMET-XL", saving_directory='/ckpts/llm/xcomet-xl')

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [29]:
model = load_from_checkpoint(model_path)

tokenizer_config.json:   0%|          | 0.00/405 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

Encoder model frozen.
/opt/nemo_rl_venv/lib/python3.12/site-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


In [30]:
data = [
    {
        "src": "Boris Johnson teeters on edge of favour with Tory MPs", 
        "mt": "Boris Johnson ist bei Tory-Abgeordneten völlig in der Gunst", 
        "ref": "Boris Johnsons Beliebtheit bei Tory-MPs steht auf der Kippe"
    }
]
model_output = model.predict(data, batch_size=8, gpus=1)
# Segment-level scores
print (model_output.scores)

# System-level score
print (model_output.system_score)

# Score explanation (error spans)
print (model_output.metadata.error_spans)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA RTX 6000 Ada Generation') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00,  3.13it/s]


[0.45751047134399414]
0.45751047134399414
[[{'text': 'ist bei', 'confidence': 0.4095492959022522, 'severity': 'critical', 'start': 13, 'end': 21}, {'text': 'Abgeordnete', 'confidence': 0.2736637592315674, 'severity': 'major', 'start': 27, 'end': 38}, {'text': 'völlig in der Gunst', 'confidence': 0.5219241380691528, 'severity': 'critical', 'start': 39, 'end': 59}]]


# test data loading

In [21]:
import numpy as np
import pandas as pd

In [3]:
import pandas as pd

df = pd.read_parquet("/data/asr/yodas/npy/parakeet-tdt-0.6b-v2_robust60-1120_langid_zh_blaser2.0-qe3.0_metricx-qe4.0_simalign/en000/manifest.parquet")

In [19]:
data = df.sample(1, random_state=42).iloc[0].to_dict()

In [23]:
np.load(data['audio_npy_path'], mmap_mode='r')[420].shape

(840, 3584)

In [20]:
data

{'audio_npy_path': '/data/asr/yodas/npy/parakeet-tdt-0.6b-v2_robust60-1120_langid_zh_blaser2.0-qe3.0_metricx-qe4.0_simalign/en000/000.npy',
 'audio_npy_row': 420,
 'audio_duration': 67.2,
 'chunk_frame_size': 14,
 'segment_info': array([{'end': 3.600000000000364, 'start': 0.3200000000001637},
        {'end': 17.04000000000042, 'start': 4.2400000000002365},
        {'end': 24.96000000000049, 'start': 17.360000000000127},
        {'end': 30.720000000000255, 'start': 25.840000000000146},
        {'end': 37.20000000000027, 'start': 31.04000000000042},
        {'end': 57.2800000000002, 'start': 38.16000000000031},
        {'end': 61.92000000000007, 'start': 57.92000000000007}],
       dtype=object),
 'src_segments': array(['Henry VIII was co-written with John Fletcher.',
        "Brian Vickers suggests that Titus Andronicus was co-written with George Peel, though Jonathan Bate, the play's most recent editor for the ardent Shakespeare, believes it to be wholly the work of Shakespeare.",
    